# 02 — Proof of Concept : Modèle MMM sur un seul marché (France)

**Auteur :** Samir EL AISSAOUY  
**Objectif :** Entraîner et évaluer le modèle bayésien sur le marché FR

Pipeline :
1. Chargement et préparation des données FR
2. Feature engineering (adstock + saturation)
3. Entraînement BayesianMMM (mode OLS fallback si PyMC absent)
4. Prédictions et métriques
5. Contributions par canal
6. ROI par canal

---

## 1. Imports & Configuration

In [5]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from src.data.data_loader     import load_market_data, split_train_test
from src.data.feature_engineering import full_feature_pipeline
from src.models.bayesian_mmm  import BayesianMMM
from src.evaluation.metrics   import compute_all_metrics, print_metrics_report
from src.utils.visualization  import (
    plot_actual_vs_predicted,
    plot_channel_contributions,
    plot_roi_comparison,
    plot_saturation_curves,
)

MARKET = 'FR'
print(f'✅ Imports OK — Marché cible : {MARKET}')

✅ Imports OK — Marché cible : FR


## 2. Chargement des données

In [6]:
df = load_market_data(MARKET)
df_train, df_test = split_train_test(df, test_ratio=0.2)

print(f'Données totales : {len(df)} semaines')
print(f'Train : {len(df_train)} semaines ({df_train["date"].min().date()} → {df_train["date"].max().date()})')
print(f'Test  : {len(df_test)}  semaines ({df_test["date"].min().date()} → {df_test["date"].max().date()})')
df.head()

Données totales : 208 semaines
Train : 167 semaines (2020-01-06 → 2023-03-13)
Test  : 41  semaines (2023-03-20 → 2023-12-25)


,market,date,week,revenue,tv_spend,facebook_spend,search_spend,ooh_spend,print_spend,competitor_price,events,trend,seasonality,promotions
0,FR,2020-01-06,1,549916.39,20253.68,14048.77,8836.37,6995.42,5298.53,98.20,0.0,1.0004,0.7500,1.0
1,FR,2020-01-13,2,623187.41,34447.24,16309.48,14416.87,9491.60,6668.70,101.97,0.0,1.0008,0.7758,0.0
2,FR,2020-01-20,3,635235.82,25088.43,16680.11,8919.51,10110.19,4566.77,108.21,0.0,1.0011,0.8037,0.0
3,FR,2020-01-27,4,716027.45,33652.54,12407.94,16185.98,8113.74,8277.64,96.06,0.0,1.0015,0.8326,0.0
4,FR,2020-02-03,5,738705.30,23330.37,14739.44,16294.06,9837.53,6335.75,105.21,1.0,1.0019,0.8609,0.0


## 3. Feature Engineering

In [7]:
df_transformed, info = full_feature_pipeline(df_train, normalize=False)

print(f'Features générées : {info["n_features"]} colonnes saturées')
print(f'Config adstock    : {list(info["adstock_config"].keys())}')

# Visualisation des courbes de saturation
fig = plot_saturation_curves(df_train)
plt.show()

TypeError: HillSaturation.__init__() got an unexpected keyword argument 'S'. Did you mean 's'?

## 4. Entraînement du modèle

In [ ]:
model = BayesianMMM(market=MARKET)

# Lance MCMC si PyMC disponible, sinon OLS fallback
model.fit(
    df_train,
    draws=1000,
    tune=1000,
    chains=4,
    random_seed=42,
)

print(f'\nModèle : {repr(model)}')

## 5. Métriques de performance

In [ ]:
# Sur le train
y_pred_train = model.predict(df_train)
metrics_train = compute_all_metrics(df_train['revenue'].values, y_pred_train)
print_metrics_report(metrics_train, label=f'Train [{MARKET}]')

# Sur le test
y_pred_test = model.predict(df_test)
metrics_test = compute_all_metrics(df_test['revenue'].values, y_pred_test)
print_metrics_report(metrics_test, label=f'Test [{MARKET}]')

In [ ]:
# Graphique réel vs prédit (test set)
fig = plot_actual_vs_predicted(df_test, y_pred_test, market=MARKET)
plt.show()

## 6. Décomposition des contributions

In [ ]:
contributions = model.get_contributions(df_train)
print('Colonnes disponibles :', contributions.columns.tolist())
contributions.head()

In [ ]:
# Contributions moyennes par canal
fig = plot_channel_contributions(contributions, market=MARKET)
plt.show()

# Résumé chiffré
channels = ['base', 'tv', 'facebook', 'search', 'ooh', 'print']
contrib_summary = contributions[[c for c in channels if c in contributions.columns]].mean()
print('\nContribution moyenne hebdomadaire :')
for canal, val in contrib_summary.items():
    pct = val / contrib_summary.sum() * 100
    print(f'  {canal:12s} : €{val:>10,.0f}  ({pct:.1f}%)')

## 7. ROI par canal

In [ ]:
roi_df = model.get_roi(df_train)
print(roi_df.to_string(index=False))

fig = plot_roi_comparison(roi_df)
plt.show()

## 8. Résumé exécutif

In [ ]:
summary = model.summary(df_train)
print(f'Marché         : {summary["market"]}')
print(f'Observations   : {summary["n_obs"]}')
print(f'Meilleur canal : {summary["best_channel"]}')
print(f'\nR²   train : {metrics_train["r2"]:.3f}')
print(f'MAPE train : {metrics_train["mape"]:.1f}%')
print(f'R²   test  : {metrics_test["r2"]:.3f}')
print(f'MAPE test  : {metrics_test["mape"]:.1f}%')